# 01 - MoMA data profiling (EDA)
Goal: understand structure, keys, quality problems and the multi-artist structure **before** cleaning.
Sources: `Artworks.csv`, `Artists.csv`, `MoMA_OnView.xlsx` (artists + artworks sheets).

In [1]:
import re, pandas as pd, numpy as np
from pathlib import Path
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 80)
RAW = Path("../raw_data")
# all as text first: we want to see raw problems, not have pandas hide them
aw = pd.read_csv(RAW/"Artworks.csv", dtype=str, keep_default_na=False, encoding="utf-8-sig")
ar = pd.read_csv(RAW/"Artists.csv",  dtype=str, keep_default_na=False, encoding="utf-8-sig")
ov = pd.read_excel(RAW/"MoMA_OnView.xlsx", sheet_name=None, dtype=str, keep_default_na=False)
ov_aw, ov_ar = ov["Artworks"], ov["Artists"]
for n, d in {"Artworks": aw, "Artists": ar, "OnView.Artworks": ov_aw, "OnView.Artists": ov_ar}.items():
    print(f"{n:16s} rows={len(d):>7,} cols={d.shape[1]}")

Artworks         rows=157,630 cols=30
Artists          rows= 15,638 cols=9
OnView.Artworks  rows=  1,142 cols=12
OnView.Artists   rows=    661 cols=7


## 1. Null / blank profile
CSV blanks are empty strings. Numeric columns use blank = unknown.

In [2]:
def blank_profile(d):
    s = d.apply(lambda col: (col.astype(str).str.strip() == "").sum())
    return pd.DataFrame({"blank": s, "blank_%": (s/len(d)*100).round(1), "distinct": d.nunique()}).sort_values("blank_%", ascending=False)
blank_profile(aw)

,blank,blank_%,distinct
Seat Height (cm),157630,100.0,1
Circumference (cm),157620,100.0,10
Weight (kg),157334,99.8,212
Length (cm),156905,99.5,393
OnView,156420,99.2,74
Diameter (cm),156161,99.1,517
Duration (sec.),155677,98.8,690
Depth (cm),139584,88.6,1398
ImageURL,66587,42.2,90408
URL,57399,36.4,100232


In [3]:
blank_profile(ar)

,blank,blank_%,distinct
ULAN,12707,81.3,2929
Wiki QID,12391,79.2,3243
Gender,3257,20.8,7
Nationality,2496,16.0,132
ArtistBio,2200,14.1,7721
ConstituentID,0,0.0,15638
DisplayName,0,0.0,15581
EndDate,0,0.0,173
BeginDate,0,0.0,237


## 2. Keys and duplicates

In [4]:
print("Artworks.ObjectID unique:", aw.ObjectID.is_unique, "| dup rows:", aw.duplicated().sum())
print("Artists.ConstituentID unique:", ar.ConstituentID.is_unique, "| dup rows:", ar.duplicated().sum())
print("Artists with duplicate DisplayName:", ar.DisplayName.duplicated(keep=False).sum())
print("Artworks with blank ConstituentID:", (aw.ConstituentID.str.strip() == "").sum())
print("Duplicate AccessionNumber:", aw.AccessionNumber.duplicated().sum())

Artworks.ObjectID unique: True | dup rows: 0
Artists.ConstituentID unique: True | dup rows: 0
Artists with duplicate DisplayName: 81
Artworks with blank ConstituentID: 1269
Duplicate AccessionNumber: 0


## 3. Multi-artist structure
`ConstituentID` / `Artist` are comma-separated. `Nationality`, `BeginDate`, `EndDate`, `Gender` are **space-separated, parenthesised** and positional, e.g. `(American) ()`. Empty `()` keeps the position. Must verify the lists line up.

In [5]:
ids = aw.ConstituentID.str.split(r",\s*")
aw["n_ids"] = ids.apply(lambda x: 0 if x == [""] else len(x))
print(aw.n_ids.value_counts().sort_index().head(12))
print("multi-artist works:", (aw.n_ids > 1).sum(), f"({(aw.n_ids>1).mean():.1%})")
paren = lambda s: re.findall(r"\(([^)]*)\)", s)
aw["n_nat"] = aw.Nationality.apply(lambda s: len(paren(s)))
aw["n_gen"] = aw.Gender.apply(lambda s: len(paren(s)))
aw["n_bd"]  = aw.BeginDate.apply(lambda s: len(paren(s)))
mis = aw[(aw.n_ids > 0) & ((aw.n_ids != aw.n_nat) | (aw.n_ids != aw.n_gen) | (aw.n_ids != aw.n_bd))]
print("rows where positional lists do NOT line up with ConstituentID count:", len(mis))
mis[["ObjectID","Artist","ConstituentID","Nationality","Gender","BeginDate"]].head(10)

n_ids
0       1269
1     148445
2       4814
3       1851
4        305
5        264
6        268
7        125
8         60
9         17
10        28
11        23
Name: count, dtype: int64
multi-artist works: 7916 (5.0%)


rows where positional lists do NOT line up with ConstituentID count: 0


,ObjectID,Artist,ConstituentID,Nationality,Gender,BeginDate


In [6]:
# artist IDs used in Artworks but missing from Artists
used = set(i for lst in ids for i in lst if i)
known = set(ar.ConstituentID)
print("distinct artist IDs in Artworks:", len(used), "| in Artists table:", len(known))
print("used but not in Artists:", len(used - known), "| in Artists but never used:", len(known - used))
print("works with no artist ID:", (aw.n_ids == 0).sum())
aw[aw.n_ids == 0][["Title","Artist","Classification","Department"]].head()

distinct artist IDs in Artworks: 14423 | in Artists table: 15638
used but not in Artists: 0 | in Artists but never used: 1215
works with no artist ID: 1269


,Title,Artist,Classification,Department
1355,Mat,,Design,Architecture & Design
1930,Bread Knife,,Design,Architecture & Design
1940,Combination Meat Chopper and Tenderizer,,Design,Architecture & Design
2108,Padlocks,,Design,Architecture & Design
2470,Wine Bottle Stand,,Design,Architecture & Design


## 4. Artists table anomalies

In [7]:
for col in ["Nationality", "Gender"]:
    print(ar[col].replace("", "<blank>").value_counts(dropna=False).head(15), "\n")
by = pd.to_numeric(ar.BeginDate, errors="coerce"); dy = pd.to_numeric(ar.EndDate, errors="coerce")
print("BeginDate == 0:", (by == 0).sum(), "| EndDate == 0:", (dy == 0).sum(), "| BeginDate blank:", by.isna().sum())
print("BeginDate range (non-zero):", by[by > 0].min(), "-", by[by > 0].max(), "| EndDate range (non-zero):", dy[dy > 0].min(), "-", dy[dy > 0].max())
print("EndDate < BeginDate:", ((dy > 0) & (by > 0) & (dy < by)).sum())
print("Lifespan > 110y:", ((dy > 0) & (by > 0) & (dy - by > 110)).sum())
ar[(dy > 0) & (by > 0) & ((dy < by) | (dy - by > 110))].head(10)

Nationality
American     5305
<blank>      2496
German        984
British       872
French        864
Italian       539
Japanese      510
Swiss         298
Dutch         282
Russian       261
Austrian      242
Canadian      194
Brazilian     175
Spanish       159
Mexican       154
Name: count, dtype: int64 

Gender
male                     9940
<blank>                  3257
female                   2435
non-binary                  3
gender non-conforming       1
transgender woman           1
female (transwoman)         1
Name: count, dtype: int64 

BeginDate == 0: 3558 | EndDate == 0: 10035 | BeginDate blank: 0
BeginDate range (non-zero): 1181 - 2017 | EndDate range (non-zero): 1226 - 2024
EndDate < BeginDate: 0
Lifespan > 110y: 2


,ConstituentID,DisplayName,ArtistBio,Nationality,Gender,BeginDate,EndDate,Wiki QID,ULAN
12271,38294,"Union Paper Bag Machine Company, Philadelphia, PA",est. 1869,,,1869,1999,,
12571,40434,"Elektrisk Bureau, Oslo",1882 – 1993,,,1882,1993,,


In [8]:
print("Nationality blank:", (ar.Nationality == "").sum(), "| ArtistBio blank:", (ar.ArtistBio == "").sum())
print("Nationality distinct:", ar.Nationality.nunique())
ar.Nationality.value_counts().tail(15)

Nationality blank: 2496 | ArtistBio blank: 2200
Nationality distinct: 132


Nationality
Sahrawi            1
Afghan             1
Ugandan            1
Burkinabé          1
Beninese           1
Persian            1
Coptic             1
Sri Lankan         1
Sierra Leonean     1
Hunkpapa Lakota    1
Caribbean          1
Indonesian         1
Ivatan             1
Nepali             1
Oneida             1
Name: count, dtype: int64

## 5. Dates
`Date` (creation) is free text. `DateAcquired` should be ISO.

In [9]:
d = aw.Date
nd = d.str.lower().str.contains(r"n\.?d")
print("blank:", (d == "").sum(), "| 'n.d.':", nd.sum())
simple = d.str.fullmatch(r"\d{4}")
print("exact 4-digit year:", simple.sum(), f"({simple.mean():.1%})")
rng = d.str.contains(r"^\d{4}\s*[^\d\s]\s*\d{2,4}$")
print("range like 1948-51 / 1948-1951:", rng.sum())
circa = d.str.contains(r"c\.|ca\.|circa", case=False)
print("circa:", circa.sum())
other = ~(simple | rng | circa | (d == "") | nd)
print("other formats:", other.sum())
d[other].sample(min(20, other.sum()), random_state=1).tolist()

blank: 2015 | 'n.d.': 2347


exact 4-digit year: 97465 (61.8%)


range like 1948-51 / 1948-1951: 29447


circa: 9943
other formats: 16420


['April, 1864',
 '(1974)',
 '1944-45?',
 '1972, published 1973',
 '(1989)',
 '1945–48, published 1948',
 '(1988)',
 'March 12, 1974',
 '1870s-80s',
 '1971, published 1972',
 '(newspaper published: May 30, 1995)',
 'April–May 1896',
 '(1954).  (Reproduced drawings executed 1951).',
 'January 1983',
 'Paris 1928-29',
 '(2008-10)',
 '(1919)',
 '(1910-11, published 1912)',
 '(2001)',
 'May - June 1991']

In [10]:
yr = pd.to_numeric(d.str.extract(r"(\d{4})")[0], errors="coerce")
print("extractable year:", yr.notna().sum(), f"({yr.notna().mean():.1%})")
print("year range:", yr.min(), "-", yr.max())
print(yr[(yr < 1800) | (yr > 2026)].value_counts().sort_index().head(20))

extractable year: 153164 (97.2%)
year range: 1768.0 - 2024.0
0
1768.0     5
1786.0     1
1797.0    80
1799.0     1
Name: count, dtype: int64


In [11]:
da = pd.to_datetime(aw.DateAcquired.replace("", np.nan), errors="coerce")
print("DateAcquired blank/unparseable:", da.isna().sum(), f"({da.isna().mean():.1%})")
print("range:", da.min(), "->", da.max(), "| before MoMA founding (1929):", (da < "1929-01-01").sum())
print("acquired in the future:", (da > pd.Timestamp.today()).sum())
print(da.dt.year.value_counts().sort_index().tail(15))
print("acquired before created:", (da.dt.year < yr).sum())

DateAcquired blank/unparseable: 6749 (4.3%)
range: 1929-11-19 00:00:00 -> 2024-10-28 00:00:00 | before MoMA founding (1929): 0
acquired in the future: 0
DateAcquired
2010.0    2050
2011.0    1916
2012.0    2943
2013.0    3109
2014.0    2829
2015.0    3817
2016.0    1994
2017.0    1620
2018.0    2507
2019.0    2286
2020.0    2600
2021.0     920
2022.0    1218
2023.0     468
2024.0     862
Name: count, dtype: int64
acquired before created: 90


## 6. Categoricals

In [12]:
for col in ["Department", "Classification", "Cataloged"]:
    print(aw[col].replace("", "<blank>").value_counts().to_string(), "\n")

Department
Drawings & Prints                        80420
Architecture & Design                    33612
Photography                              33608
Painting & Sculpture                      4055
Media and Performance                     3198
Fluxus Collection                         1683
Film                                      1024
Architecture & Design - Image Archive       30 

Classification
Photograph                        34770
Print                             32590
Illustrated Book                  27834
Mies van der Rohe Archive         16305
Drawing                           14234
Design                            12307
Architecture                       4158
Painting                           2430
Video                              2420
Notebook                           1965
Sculpture                          1787
Multiple                           1182
Periodical                          951
Installation                        907
Frank Lloyd Wright Archive          

In [13]:
med = aw.Medium.replace("", np.nan)
print("Medium blank:", med.isna().sum(), "| distinct:", med.nunique())
print("share of works covered by top 25 mediums:", med.value_counts().head(25).sum() / med.notna().sum())
med.value_counts().head(25)

Medium blank:

 9247 | distinct: 22777
share of works covered by top 25 mediums: 0.408860853332255


Medium
Gelatin silver print                       17012
Lithograph                                  8633
Pencil on paper                             7233
Albumen silver print                        4837
Pencil on tracing paper                     2603
Chromogenic print                           1744
Letterpress                                 1706
Etching                                     1685
Ink on paper                                1564
Pencil and colored pencil on paper          1262
Offset lithograph                           1151
Oil on canvas                               1071
Drypoint                                    1058
Woodcut                                     1000
Video (color, sound)                         981
Screenprint                                  937
Gelatin silver printing-out-paper print      915
Inkjet print                                 909
Lithograph, printed in color                 845
Pencil on acetate                            658
Poster       

In [14]:
print("Credit line leading words:"); print(aw.CreditLine.str.extract(r"^(\w+(?:\s\w+)?)")[0].value_counts().head(12))

Credit line leading words:


0
Gift of              61719
Mies van             15504
The Louis            11417
Purchase              8459
Acquired through      5511
The Gilbert           5453
Abbott                4910
Committee on          3858
The Judith            2593
Fund for              1935
Given anonymously     1531
Anonymous gift        1478
Name: count, dtype: int64


## 7. Physical dimensions & duration

In [15]:
num = ["Circumference (cm)","Depth (cm)","Diameter (cm)","Height (cm)","Length (cm)","Weight (kg)","Width (cm)","Seat Height (cm)","Duration (sec.)"]
nm = aw[num].apply(pd.to_numeric, errors="coerce")
pd.DataFrame({"filled": nm.notna().sum(), "filled_%": (nm.notna().mean()*100).round(1), "zeros": (nm == 0).sum(), "min": nm.min(), "median": nm.median(), "p99": nm.quantile(.99), "max": nm.max()})

,filled,filled_%,zeros,min,median,p99,max
Circumference (cm),10,0.0,0,9.900,36.0000,83.573218,8.380000e+01
Depth (cm),18046,11.4,10373,0.000,0.0000,259.080000,1.808484e+03
Diameter (cm),1469,0.9,0,0.635,13.8000,196.112000,9.144000e+02
Height (cm),128219,81.3,2451,0.000,27.8000,203.888373,9.140000e+03
Length (cm),725,0.5,58,0.000,27.0000,643.322588,8.321057e+03
Weight (kg),296,0.2,0,0.090,23.3602,26778.894130,1.850676e+05
Width (cm),127293,80.8,2328,0.000,25.4000,207.010401,9.144000e+03
Seat Height (cm),0,0.0,0,NaN,NaN,NaN,NaN
Duration (sec.),1953,1.2,217,0.000,420.0000,41328.000000,6.283065e+06


In [16]:
print("Height & Width filled by Classification (% of works):")
t = aw.assign(has_hw=nm["Height (cm)"].notna() & nm["Width (cm)"].notna()).groupby("Classification").agg(works=("ObjectID","count"), hw_pct=("has_hw", lambda s: round(s.mean()*100,1)))
t.sort_values("works", ascending=False)

Height & Width filled by Classification (% of works):


,works,hw_pct
Classification,,
Photograph,34770,81.7
Print,32590,90.6
Illustrated Book,27834,97.5
Mies van der Rohe Archive,16305,20.9
Drawing,14234,97.3
Design,12307,74.7
Architecture,4158,90.3
Painting,2430,98.6
Video,2420,3.2


In [17]:
print("Duration populated by Classification:")
aw.assign(has_dur=nm["Duration (sec.)"].notna()).groupby("Classification").has_dur.sum().sort_values(ascending=False).head(8)

Duration populated by Classification:


Classification
Video           1665
Installation     160
Film              72
Architecture      32
Design            12
Photograph         3
Audio              3
Performance        1
Name: has_dur, dtype: int64

## 8. Cataloged and OnView

In [18]:
aw["onview_flag"] = aw.OnView.str.strip() != ""
print("on view in CSV:", aw.onview_flag.sum(), f"({aw.onview_flag.mean():.2%})")
print(pd.crosstab(aw.Cataloged, aw.onview_flag))
print(aw[aw.onview_flag].OnView.value_counts().head(10))

on view in CSV: 1210 (0.77%)
onview_flag  False  True 
Cataloged                
N            57358     41
Y            99062   1169
OnView
"MoMA, Floor 3, 3 East"     128
"MoMA, Floor 1, 1 South"     47
"MoMA, Floor 5, 503"         46
"MoMA, Floor 3, 3 South"     41
"MoMA, Floor 5, 501"         40
"MoMA, Floor 2, 216"         40
"MoMA, Floor 4, 417"         35
"MoMA, Floor 2, 2 South"     34
"MoMA, Floor 5, 521"         34
"MoMA, Floor 5, 514"         34
Name: count, dtype: int64


In [19]:
print("OnView.Artworks distinct ObjectIDs:", ov_aw.ObjectID.nunique(), "of", len(ov_aw))
xl, cs = set(ov_aw.ObjectID), set(aw[aw.onview_flag].ObjectID)
print("in xlsx only:", len(xl - cs), "| in CSV-OnView only:", len(cs - xl), "| both:", len(xl & cs))
print("xlsx ObjectIDs missing from Artworks.csv entirely:", len(xl - set(aw.ObjectID)))
print("xlsx OnView blank:", (ov_aw.OnView.str.strip() == "").sum())
print(ov_aw.OnView.str.extract(r"Floor (\d)")[0].value_counts(dropna=False).sort_index())

OnView.Artworks distinct ObjectIDs: 1142 of 1142
in xlsx only: 0 | in CSV-OnView only: 68 | both: 1142
xlsx ObjectIDs missing from Artworks.csv entirely: 0
xlsx OnView blank: 0


0
1       73
2      135
3      151
4      275
5      505
6        2
NaN      1
Name: count, dtype: int64


In [20]:
# do shared works agree on location?
j = ov_aw[["ObjectID","OnView"]].merge(aw[["ObjectID","OnView"]], on="ObjectID", suffixes=("_xlsx","_csv"))
print("shared:", len(j), "| location equal:", (j.OnView_xlsx == j.OnView_csv).sum())
j[j.OnView_xlsx != j.OnView_csv].head(10)

shared: 1142 | location equal: 1142


,ObjectID,OnView_xlsx,OnView_csv


In [21]:
print("xlsx artworks with comma ConstituentID:", ov_aw.ConstituentID.str.contains(",").sum())
xi = set(i for s in ov_aw.ConstituentID for i in re.split(r",\s*", s) if i)
print("xlsx artist IDs used:", len(xi), "| in xlsx Artists:", len(xi & set(ov_ar.ConstituentID)), "| in Artists.csv:", len(xi & set(ar.ConstituentID)))
print("xlsx Artists subset of Artists.csv:", set(ov_ar.ConstituentID) <= set(ar.ConstituentID))

xlsx artworks with comma ConstituentID: 52
xlsx artist IDs used: 643 | in xlsx Artists: 643 | in Artists.csv: 643
xlsx Artists subset of Artists.csv: True


## 9. Findings
See `docs/eda_findings.md` for the written summary of the results above, which drives `02_cleaning_and_model.ipynb`.